#### 소비자 보호원 서비스 집단 분쟁 조정 사례집 RAG

In [35]:
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ibm import WatsonxEmbeddings
from langchain_chroma import Chroma

from pydantic import BaseModel, Field
from langchain_core.runnables import RunnablePassthrough

import re
from pprint import pprint

from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers import (
    EnsembleRetriever, 
    ContextualCompressionRetriever, 
    BM25Retriever
)
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, EmbeddingsFilter, DocumentCompressorPipeline

In [2]:
#.env 내용 가죠오기
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")
cohere_api_key = os.getenv("COHERE_API_KEY")

In [3]:
watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}",
    params = {
    "max_tokens": 2000,
    "temperature": 0
    }
)

watsonx_enbedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}"
    )

In [4]:
pdf_path = "./data/2018 서비스·집단 분쟁조정 사례집.pdf"

# loader 선택 - pdf loader 선택가능
loader = PyPDFLoader(pdf_path)
docs = loader.load()
len(docs)

200

In [5]:
#
pprint(docs[0].page_content)
pprint(docs[0].metadata)

'소비자분쟁조정위원회\n2018\n서비스·집단 \n분쟁조정 사례집'
{'author': 'PC_A2',
 'creationdate': '2019-06-05T11:33:24+09:00',
 'creator': 'PScript5.dll Version 5.2.2',
 'moddate': '2019-06-05T11:58:31+09:00',
 'page': 0,
 'page_label': '1',
 'producer': 'Acrobat Distiller 9.0.0 (Windows)',
 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf',
 'title': '<32303139303630355FBCD2BAF1C0DABFF820BBE7B7CAC1FD5BBCADBAF1BDBA5D5FB3BBC1F65FC6EDC1FDBABB76657231312E687770>',
 'total_pages': 200}


In [6]:
# 첫 번째 사건 가져오기
pprint(docs[10].page_content)
pprint(docs[10].metadata)

('제1장\n'
 '일\n'
 '반\n'
 '분\n'
 '쟁\n'
 '조\n'
 '정\n'
 ' 사\n'
 '례 (\n'
 '서\n'
 '비\n'
 '스 )\n'
 '제1장 일반분쟁조정 사례(서비스) ● 3\n'
 '사\n'
 '례 01 사건번호 2018일나565  | 결정일자 2018. 8. 7.\n'
 '세탁 후 갑피 마모 및 경화된 가죽 \n'
 '운동화에 대한 손해배상 요구\n'
 '주 문\n'
 '1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n'
 '운동화, 색상 : 흰색) 1켤레를 반환한다. \n'
 '2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n'
 '을 지급한다.\n'
 '이 유\n'
 '1. 기초사실\n'
 '가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n'
 '이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n'
 '피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n'
 '마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n'
 '으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n'
 '청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n'
 '다는 이유로 이를 거부하였다.\n'
 '나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.\n'
 '   \n'
 '신청인이 주장하는 갑피 벗겨짐(스크래치 등) 증상은 관찰되나 현 제품 상태만\n'
 '으로는 제품 훼손의 원인이 세탁 과정상 발생한 것인지 착화 환경에 따른 문제\n'
 '인지 단정하기 어려운바, 판단 불가하다.')
{'author': 'PC_A2',
 'creationdate': '2019-06-05T11:33:24+09:00',
 'creator'

### 전처리
- 정규식을 이용한 내용 추출
    - 수량자(반복 횟수 지정)
        - * : 0번 이상 반복(어느 상황에서든지 작동)
            ex) a*b => a, ab, aab, aaab,.....
        - + : 1번 이상 반복(최소 1번 이상 반복)
            ex) a+b = > ab, aab, .... (a, b는 매칭 안됨)
        - ? : 0 or 1 (있거나 없음)
            ex) a?b => b, ab
        - {n} : 정확히 n번 반복
            ex) \d{4} => 숫자 4자리
        - {n,m} : n번부터 m번까지 반복
            ex) \d{1,2} =>숫자 1자리 or 2자리
    - 문자 class
        - \d : 숫자와 매칭
        - \D : 숫자 이외와 매칭
        - \s : 공백 문자(spase, tab, \n)
        - \S : 공백 문자 이외와 매칭
        - \w : 문자+숫자+_와 매칭(한글, 영어, 숫자)
        - \W : 문자+숫자 이외 (특수문자, 공백)
        - . : \n을 제외한 모든 문자 1개
            - a.b => aab,a1b,axb,.....(ab는 안됨)


In [7]:
pattern = r"사\n례\s*\d+ 사건번호.*결정일자.*\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\."

split_text = re.findall(pattern,"".join(docs[10].page_content))
split_text

['사\n례 01 사건번호 2018일나565  | 결정일자 2018. 8. 7.']

In [8]:
# 패터닝 존재하면 패턴을 기준으로 문서를 분리
if split_text:
    parts = re.split(pattern, "".join(docs[10].page_content))

parts

['제1장\n일\n반\n분\n쟁\n조\n정\n 사\n례 (\n서\n비\n스 )\n제1장 일반분쟁조정 사례(서비스) ● 3\n',
 '\n세탁 후 갑피 마모 및 경화된 가죽 \n운동화에 대한 손해배상 요구\n주 문\n1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n운동화, 색상 : 흰색) 1켤레를 반환한다. \n2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n을 지급한다.\n이 유\n1. 기초사실\n가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n다는 이유로 이를 거부하였다.\n나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.\n   \n신청인이 주장하는 갑피 벗겨짐(스크래치 등) 증상은 관찰되나 현 제품 상태만\n으로는 제품 훼손의 원인이 세탁 과정상 발생한 것인지 착화 환경에 따른 문제\n인지 단정하기 어려운바, 판단 불가하다.']

In [9]:
# metadata로 사용할 정보 추출
# 사례번호, 사건번호, 결정일자 추출
# 
split_text[0] 

# 사례번호
re.findall(r"례\s?(\d+)\s?사건번호",split_text[0])[0]

# 사건번호
re.findall(r"사건번호\s+([^\s|]+)",split_text[0])[0]

# 결정일자
re.findall(r"결정일자\s+(\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\.)",split_text[0])[0]

'2018. 8. 7.'

In [10]:
class CaseMetadata(BaseModel):
    case_number : str = Field(description="사건번호 예: 2018일나565")
    decision_data : str = Field(description="결정일자 예: 2018. 8. 7.")

In [11]:
metadata_prompt= PromptTemplate.from_template(
    """
다음은 분쟁 조정 사례에 대한 텍스트입니다.
- case_number : 사건번호
- decision_data : 결정일자

반드시 json으로 반환하세요
{case_text}
    """
)

structed_llm=watson_llm.with_structured_output(CaseMetadata)
chain = metadata_prompt|structed_llm
case_metadata = chain.invoke({'case_text':split_text})
print(case_metadata)
print(dict(case_metadata))

case_number='2018일나565' decision_data='2018. 8. 7.'
{'case_number': '2018일나565', 'decision_data': '2018. 8. 7.'}


#### page_content 내용 추출

In [12]:
# 주 문 ~~~~~~~~~~~~~
parts[1]

# 주 문 위치 찾기
re.search(r"주 문\n",parts[1]).span()

# 제목 추출
title = parts[1][:re.search(r"주 문\n",parts[1]).span()[0]].strip()
# 내용 추출
content = parts[1][re.search(r"주 문\n",parts[1]).span()[0]:].strip()

In [13]:
pprint(title)
pprint(content)

'세탁 후 갑피 마모 및 경화된 가죽 \n운동화에 대한 손해배상 요구'
('주 문\n'
 '1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n'
 '운동화, 색상 : 흰색) 1켤레를 반환한다. \n'
 '2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n'
 '을 지급한다.\n'
 '이 유\n'
 '1. 기초사실\n'
 '가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n'
 '이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n'
 '피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n'
 '마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n'
 '으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n'
 '청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n'
 '다는 이유로 이를 거부하였다.\n'
 '나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.\n'
 '   \n'
 '신청인이 주장하는 갑피 벗겨짐(스크래치 등) 증상은 관찰되나 현 제품 상태만\n'
 '으로는 제품 훼손의 원인이 세탁 과정상 발생한 것인지 착화 환경에 따른 문제\n'
 '인지 단정하기 어려운바, 판단 불가하다.')


In [14]:
# 사례번호,사건번호,결정일자,제목은 metadata로 추가
# 내용 page_content update

pdf_docs = []
case_metadata = {}

# 사건이 시작되는 페이지 ~ 마지막에서 -2 페이지까지 반복
for doc in docs[10:-2]:

    split_text = re.findall(pattern, "".join(doc.page_content))
    if split_text:
        
        # 사례번호 추출
        case_metadata["case_id"] = re.findall(r"례\s?(\d+)\s?사건번호",split_text[0])[0]

        # 패턴 기준으로 텍스트 분할
        parts = re.split(pattern, "".join(doc.page_content))

        if re.search(r"주 문\n",parts[1]):
            # 제목 추출
            case_metadata["title"] = parts[1][:re.search(r"주 문\n",parts[1]).span()[0]].replace("\n","").strip()
            # 내용 추출 후 기존 내용 업데이트
            doc.page_content = parts[1][re.search(r"주 문\n",parts[1]).span()[0]:].strip()
        else:
            case_metadata["title"] = ""

        i = 0
        while i < 10:
            try:
                # 사건번호, 결정일자 추출    
                response = chain.invoke({"case_text":split_text[0]})
                for k,v in dict(response).items():
                    case_metadata[k] = v.replace("\n","").replace(" ","")
                break  
            
            except:
                i += 1
                continue

        doc.metadata.update(case_metadata)       
        
        pdf_docs.append(doc)
    else:
        doc.metadata.update(case_metadata)
        pdf_docs.append(doc)

len(pdf_docs)       

188

In [15]:
pdf_docs[10].metadata
pprint(pdf_docs[10].page_content)

('제1장\n'
 '일\n'
 '반\n'
 '분\n'
 '쟁\n'
 '조\n'
 '정\n'
 ' 사\n'
 '례 (\n'
 '서\n'
 '비\n'
 '스 )\n'
 '제1장 일반분쟁조정 사례(서비스) ● 13\n'
 '살피건대, 신청인의 출국일인 2017. 12. 18. 인천공항에 강설로 인한 활주로 제빙작\n'
 '업 등이 있었는지 여부에 대해 인천국제공항공사 직원(정○○)에게 문의한 결과, 실제\n'
 '로 동 작업을 실시하였음이 확인되었고, 더욱이 같은 날 인천공항을 이용하였던 타 이\n'
 '용자들의 후기(http:// ∆∆∆∆∆.tistory.com/21)에서도 강설로 인한 제빙작업으로 \n'
 '2시간 이상 지연된 사실이 확인되는 바, 기상상황으로 인하여 지연되었다는 피신청인\n'
 '의 주장은 인정된다.\n'
 '한편, 신청인은 이 사건 항공편보다 늦게 출발이 예정된 나고야행 항공편(○○742편)\n'
 '이 먼저 출발하였다고 주장하나 이에 대한 객관적인 입증자료는 존재하지 아니하고, \n'
 '피신청인은 활주로 제빙작업은 기장의 요청에 의하여 진행될 수도 있는데, 위 항공편\n'
 '의 경우 제빙작업 없이 바로 출발하였기 때문에 먼저 출발할 수 있었다고 항변하는바, \n'
 '이 사건 항공편의 지연과는 무관하다고 봄이 상당하며, 「소비자기본법」에 따른 「소비\n'
 '자분쟁해결기준」은 기상 및 공항사정으로 인한 지연의 경우 항공사의 손해배상 사유에\n'
 '서 제외하고 있고, 이 사건 운송지연의 근본원인은 기상사정에 의한 것으로 확인되므\n'
 '로, 피신청인에게 운송지연에 따른 손해배상 책임이 인정된다고 보기 어렵다. \n'
 '이상을 종합할 때, 신청인과 피신청인 사이의 이 사건 분쟁조정 신청에 대하여는 「소\n'
 '비자분쟁조정규칙」제32조 제3호에 따라 조정하지 아니함이 상당하다.\n'
 '[관련 법규 및 고시] 상법 제907조, 소비자분쟁조정규칙 제32조, 소비자분쟁해결기준 \n'
 '별표Ⅱ. 품목별 보상기준 33. 운수업\n'
 '이상

### 2. 분할

In [16]:
# 500 / 50
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
split_docs = splitter.split_documents(docs)

print(split_docs[0].page_content)

소비자분쟁조정위원회
2018
서비스·집단 
분쟁조정 사례집


In [17]:
print(split_docs[1].page_content)

2018 분쟁조정  사례집
소비자 관련 분쟁은 소액이라는 특징 때문에 법원의 소송으로 해결하기 어려운 면이 있습니다. 이러한 점 때문에 
소송 외 분쟁해결기구로서 설립된 소비자분쟁조정위원회는 원만한 합의가 이루어지지 않은 소비자와 사업자에게 
객관적이고 공정한 조정안을 제시함으로써 분쟁이 합리적이고 원활히 해결될 수 있도록 노력하고 있습니다.
소비자기본법에 근거하여 1987년 설립된 소비자분쟁조정위원회는 설립 첫 해 20건의 사건 조정을 시작으로, 2004년 
이후부터는 매년 1,000건이 넘는 사건을 조정하였으며, 2018년에는 3,080여건을 처리하는 등 그 역할을 충실히 
수행하고 있습니다.
특히, 분쟁조정 사건을 신속하고 공정하게 처리하기 위해 2017년에는 소비자기본법 개정을 통해 조정위원을 50명
에서 150명으로 확대하는 등 관련 법·제도 개선으로 소비자권익증진에 기여하고 있습니다.


In [18]:
# 마침표 뒤에 나오는 나머지 줄바꿈 문자 이외의 줄바꿈 문자 제거
pprint(split_docs[18].page_content)

text = split_docs[18].page_content
text = re.sub(r"(?<!\.)\n"," ", text)
pprint(text)

('주 문\n'
 '1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n'
 '운동화, 색상 : 흰색) 1켤레를 반환한다. \n'
 '2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n'
 '을 지급한다.\n'
 '이 유\n'
 '1. 기초사실\n'
 '가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n'
 '이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n'
 '피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n'
 '마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n'
 '으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n'
 '청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n'
 '다는 이유로 이를 거부하였다.\n'
 '나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.')
('주 문 1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽  운동화, 색상 : 흰색) 1켤레를 '
 '반환한다.  2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원 을 지급한다.\n'
 '이 유 1. 기초사실 가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색,  이하 ‘이 사건 '
 '제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10.  피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 '
 '4,000원)하였는데 수령 후 갑피  마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였 으나, 이후에도 '
 '경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신 청

### Indexing - Embedding

In [19]:
from copy import deepcopy

final_docs = []

for doc in split_docs:
    new_doc = deepcopy(doc)

    text = re.sub(r"(?<!\.)\n"," ", new_doc.page_content)

    new_doc.page_content = (
        f"### 이 사건은 '{new_doc.metadata['title']} 에 대한 사례입니다.\n\n"
        f"{text}"
    )

    final_docs.append(new_doc)


In [20]:
final_docs[18].page_content

"### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.\n\n주 문 1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽  운동화, 색상 : 흰색) 1켤레를 반환한다.  2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원 을 지급한다.\n이 유 1. 기초사실 가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색,  이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10.  피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피  마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였 으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신 청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없 다는 이유로 이를 거부하였다.\n나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다."

In [21]:
# 로컬저장: Chroma
# 메모리: FAISS(로컬저장 가능)

vectorstore = Chroma.from_documents(
    documents=final_docs,embedding=watsonx_enbedding,persist_directory="./db/chroma_db",
    collection_name="customer_dispute_cases"
)

In [22]:
query = "세탁 후 오염에 대한 손해배상 책임은 어떻게 이루어지나요?"

similarity_docs = vectorstore.similarity_search(query, k=5)

pprint(similarity_docs)

[Document(id='b5fe62d8-1105-470b-baf8-9e7a86ceac02', metadata={'decision_data': '2018.8.7.', 'page_label': '12', 'total_pages': 200, 'title': '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구', 'case_number': '2018일나565', 'creationdate': '2019-06-05T11:33:24+09:00', 'creator': 'PScript5.dll Version 5.2.2', 'case_id': '01', 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'author': 'PC_A2', 'moddate': '2019-06-05T11:58:31+09:00', 'page': 11}, page_content="### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.\n\n탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품을 훼손 하여 그 손해가 발생 및 확대되었을 가능성을 배제할 수 없는 점 등에 비추어 볼 때,  손해의 공평·타당한 분담이라는 손해배상 제도의 지도이념과 상호 양보를 통한 분쟁의  원만한 해결이라는 조정의 취지를 고려하여, 피신청인의 책임을 60%로 제한함이 상당 하다.\n한편, 세탁비와 관련하여,「세탁업 표준약관」제9조 제1항 및 제2항에서는 세탁업자의  책임있는 사유로 세탁물이 손상, 색상변화, 얼룩 등의 하자가 발생였을 때에는 해당  세탁물에 대하여 세탁업자는 고객에게 세탁요금을 청구하지 못하므로, 세탁업자인 피 신청인이 세탁비 4,000원을 신청인에게 환급이 상당하다.\n이상을 종합하면, 신청인은 피신청인에게 이 사건 제품을 반환하고 피신청인은 손해배 상액 67,000원(112,140원 × 60%, 1,00

In [24]:
for doc in similarity_docs:
    print(doc.metadata['case_id'], doc.metadata['page'],doc.page_content[:500])
    print('-----------')

01 11 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품을 훼손 하여 그 손해가 발생 및 확대되었을 가능성을 배제할 수 없는 점 등에 비추어 볼 때,  손해의 공평·타당한 분담이라는 손해배상 제도의 지도이념과 상호 양보를 통한 분쟁의  원만한 해결이라는 조정의 취지를 고려하여, 피신청인의 책임을 60%로 제한함이 상당 하다.
한편, 세탁비와 관련하여,「세탁업 표준약관」제9조 제1항 및 제2항에서는 세탁업자의  책임있는 사유로 세탁물이 손상, 색상변화, 얼룩 등의 하자가 발생였을 때에는 해당  세탁물에 대하여 세탁업자는 고객에게 세탁요금을 청구하지 못하므로, 세탁업자인 피 신청인이 세탁비 4,000원을 신청인에게 환급이 상당하다.
이상을 종합하면, 신청인은 피신청인에게 이 사건 제품을 반환하고 피신청인은 손해배 상액 67,000원(112,140원 × 60%, 
-----------
01 11 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.

살피건대, 피신청인은 세탁 전부터 이 사건 제품의 상태가 좋지 않았다고 주장하나,  다음과 같은 사정들, 즉 ① 피신청인이 세탁을 위해 이 사건 제품을 인수하면서 세탁 물의 하자유무가 작성된 인수증을 교부하지 않은 점, ② 「세탁업 표준약관」 제3조 제1 항은 인수증 미교부로 인해 발생한 손해 및 그에 따른 손해배상책임은 세탁업자에게  귀속되는 것으로 규정하고 있는 점, ③ 인수 당시 이 사건 제품에 이미 하자가 있었음 을 입증할 만한 객관적인 자료가 없는 점 등에 비추어 볼 때, 피신청인의 위 주장은  이유 없으므로, 피신청인은 「소비자분쟁해결기준」에 따라 이 사건 제품의 잔존가치  112,140원(= 구매대금 160,200원 × 70%)을 배상함이 상당하다.
다만, 한국소비자원 신발제품심의위원회의 심의 결과, 이 사건 현상이 피신

In [26]:
retriever = vectorstore.as_retriever(search_kwags={'k':5, "filter":{'case_id':'01'}})

retrieved_docs = retriever.invoke(query)
for doc in retrieved_docs:
    print(doc.metadata['case_id'], doc.metadata['page'],doc.page_content[:500])
    print('-----------')

01 11 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품을 훼손 하여 그 손해가 발생 및 확대되었을 가능성을 배제할 수 없는 점 등에 비추어 볼 때,  손해의 공평·타당한 분담이라는 손해배상 제도의 지도이념과 상호 양보를 통한 분쟁의  원만한 해결이라는 조정의 취지를 고려하여, 피신청인의 책임을 60%로 제한함이 상당 하다.
한편, 세탁비와 관련하여,「세탁업 표준약관」제9조 제1항 및 제2항에서는 세탁업자의  책임있는 사유로 세탁물이 손상, 색상변화, 얼룩 등의 하자가 발생였을 때에는 해당  세탁물에 대하여 세탁업자는 고객에게 세탁요금을 청구하지 못하므로, 세탁업자인 피 신청인이 세탁비 4,000원을 신청인에게 환급이 상당하다.
이상을 종합하면, 신청인은 피신청인에게 이 사건 제품을 반환하고 피신청인은 손해배 상액 67,000원(112,140원 × 60%, 
-----------
01 11 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.

살피건대, 피신청인은 세탁 전부터 이 사건 제품의 상태가 좋지 않았다고 주장하나,  다음과 같은 사정들, 즉 ① 피신청인이 세탁을 위해 이 사건 제품을 인수하면서 세탁 물의 하자유무가 작성된 인수증을 교부하지 않은 점, ② 「세탁업 표준약관」 제3조 제1 항은 인수증 미교부로 인해 발생한 손해 및 그에 따른 손해배상책임은 세탁업자에게  귀속되는 것으로 규정하고 있는 점, ③ 인수 당시 이 사건 제품에 이미 하자가 있었음 을 입증할 만한 객관적인 자료가 없는 점 등에 비추어 볼 때, 피신청인의 위 주장은  이유 없으므로, 피신청인은 「소비자분쟁해결기준」에 따라 이 사건 제품의 잔존가치  112,140원(= 구매대금 160,200원 × 70%)을 배상함이 상당하다.
다만, 한국소비자원 신발제품심의위원회의 심의 결과, 이 사건 현상이 피신

In [ ]:
retriever = vectorstore.as_retriever(search_kwags={'k':5, "where_document":{'$contains':'세탁'}})

retrieved_docs = retriever.invoke(query)
for doc in retrieved_docs:
    print(doc.metadata['case_id'], doc.metadata['page'],doc.page_content[:500])
    print('-----------')

01 11 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품을 훼손 하여 그 손해가 발생 및 확대되었을 가능성을 배제할 수 없는 점 등에 비추어 볼 때,  손해의 공평·타당한 분담이라는 손해배상 제도의 지도이념과 상호 양보를 통한 분쟁의  원만한 해결이라는 조정의 취지를 고려하여, 피신청인의 책임을 60%로 제한함이 상당 하다.
한편, 세탁비와 관련하여,「세탁업 표준약관」제9조 제1항 및 제2항에서는 세탁업자의  책임있는 사유로 세탁물이 손상, 색상변화, 얼룩 등의 하자가 발생였을 때에는 해당  세탁물에 대하여 세탁업자는 고객에게 세탁요금을 청구하지 못하므로, 세탁업자인 피 신청인이 세탁비 4,000원을 신청인에게 환급이 상당하다.
이상을 종합하면, 신청인은 피신청인에게 이 사건 제품을 반환하고 피신청인은 손해배 상액 67,000원(112,140원 × 60%, 
-----------
01 11 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.

살피건대, 피신청인은 세탁 전부터 이 사건 제품의 상태가 좋지 않았다고 주장하나,  다음과 같은 사정들, 즉 ① 피신청인이 세탁을 위해 이 사건 제품을 인수하면서 세탁 물의 하자유무가 작성된 인수증을 교부하지 않은 점, ② 「세탁업 표준약관」 제3조 제1 항은 인수증 미교부로 인해 발생한 손해 및 그에 따른 손해배상책임은 세탁업자에게  귀속되는 것으로 규정하고 있는 점, ③ 인수 당시 이 사건 제품에 이미 하자가 있었음 을 입증할 만한 객관적인 자료가 없는 점 등에 비추어 볼 때, 피신청인의 위 주장은  이유 없으므로, 피신청인은 「소비자분쟁해결기준」에 따라 이 사건 제품의 잔존가치  112,140원(= 구매대금 160,200원 × 70%)을 배상함이 상당하다.
다만, 한국소비자원 신발제품심의위원회의 심의 결과, 이 사건 현상이 피신

#### Generation

In [32]:
def format_docs(docs):
    """Document 객체에서 oage_content 추출"""
    return "\n\n".join([d.page_content for d in docs])


retriever = vectorstore.as_retriever(search_kwags={'k':5})

rag_prompt = ChatPromptTemplate.from_messages([
    ("system","다음 컨텍스트를 참고하여 질문에 답하세요.\n컨텍스트에 없는 내용은 모른다고 답하세요\n\n컨텍스트:\n{context}"),
    ("human","{query}")
])

# 질의 => 벡터화 => 가장 가까운 chunk 찾기 => Document 객체 => format_docs => context => LLM Context 기반으로 답변 정리

chain = {
    "context":retriever | format_docs,
    "query":RunnablePassthrough()
    } | rag_prompt | watson_llm | StrOutputParser()

query = "세탁 후 오염에 대한 손해배상 책임은 어떻게 이루어지나요?"

response = chain.invoke(query)

In [33]:
response

'세탁 후 오염에 대한 손해배상 책임은 세탁업자와 고객 간의 계약 조건, 세탁업자의 책임, 그리고 고객의 행위 등에 따라 다르게 적용될 수 있습니다. 일반적으로 세탁업자는 고객의 물건을 세탁하는 과정에서 발생하는 손해에 대해 책임을 지게 됩니다. 그러나 이러한 책임은 세탁업자가 고객의 물건을 세탁하는 과정에서 합리적인 주의를 기울였다는 것을 증명할 수 있다면 제한될 수 있습니다.\n\n세탁업자의 책임은 다음과 같은 경우에 제한될 수 있습니다:\n\n1. **고객의 과실**: 고객이 세탁 전에 물건을 적절히 준비하지 않았거나, 세탁업자에게 물건의 특성이나 손상 가능성에 대해 알리지 않았을 경우, 고객의 과실로 인해 발생한 손해에 대해 세탁업자는 책임을 지지 않을 수 있습니다.\n\n2. **물건의 특성**: 특정 물건이 세탁 과정에서 손상되기 쉬운 특성을 가지고 있다면, 세탁업자는 이러한 특성을 고객에게 사전에 알려주어야 하며, 고객이 이를 인지하고 세탁을 진행한 경우 세탁업자의 책임이 제한될 수 있습니다.\n\n3. **계약 조건**: 세탁업자와 고객 간의 계약에 특정 조건이 명시되어 있다면, 이러한 조건에 따라 세탁업자의 책임이 제한되거나 변경될 수 있습니다. 예를 들어, 세탁업자가 일정 금액 이상의 손해에 대해서는 책임지지 않는다는 조건이 명시되어 있다면, 이에 따라 책임이 제한될 수 있습니다.\n\n4. **법률 규정**: 일부 국가나 지역에서는 세탁업자의 책임을 제한하는 법률 규정이 존재할 수 있습니다. 이러한 법률 규정에 따라 세탁업자의 책임이 제한될 수 있습니다.\n\n결론적으로, 세탁 후 오염에 대한 손해배상 책임은 세탁업자와 고객 간의 구체적인 상황에 따라 다르게 적용될 수 있으며, 세탁업자의 책임을 판단할 때는 세탁업자의 행위, 고객의 행위, 그리고 계약 조건 등을 종합적으로 고려해야 합니다.'

### BM25 (Sparse Retrieval)
- 키워드 검색

In [36]:
bm25_retriever = BM25Retriever.from_documents(final_docs)
bm25_retriever.k = 5

docs = bm25_retriever.invoke("가죽 운동화 세탁 보상방법")

for doc in docs:
    print(doc.metadata['case_id'],doc.page_content[:500])
    print('-----------')

01 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.

4 ● 2018 서비스·집단 분쟁조정 사례집 2 .  판   단 신청인은 피신청인의 세탁 후 신발의 갑피가 마모되고 경화되는 현상이 발생하였고,  재세탁 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않았으며, 이  사건 제품보다 더 오래신은 비슷한 흰색 가죽 운동화의 상태보다 이 사건 제품의 상태 가 더 좋지 않기 때문에 본인의 착화습관이 이 사건 현상의 원인이라고 볼 수도 없으 므로, 세탁과실에 따른 손해배상 및 세탁비 환급을 요구한다.
이에 대하여 피신청인은 이 사건 제품을 인수하였을 당시 이미 제품 상태가 좋지 않았 고, 한국소비자원의 신발제품심의위원회에서도 세탁과실로 인정하지 않았기 때문에 신 청인의 요구를 수용할 수 없으나, 다만 원만한 해결을 위해 피해구제 담당자가 제시한  배상산정액 112,140원의 50%인 50,670원을 환급할 의사는 있다고 
-----------
01 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.

나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.
    신청인이 주장하는 갑피 벗겨짐(스크래치 등) 증상은 관찰되나 현 제품 상태만 으로는 제품 훼손의 원인이 세탁 과정상 발생한 것인지 착화 환경에 따른 문제 인지 단정하기 어려운바, 판단 불가하다.
-----------
01 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.

살피건대, 피신청인은 세탁 전부터 이 사건 제품의 상태가 좋지 않았다고 주장하나,  다음과 같은 사정들, 즉 ① 피신청인이 세탁을 위해 이 사건 제품을 인수하면서 세탁 물의 하자유무가 작성된 인수증을 교부하지 않은 점, ② 「세탁업 표준약관」 제3조 제1 항은 인수증 미교부로 인해 발생한 손해 및 그에 따른 손해배상책임은 세탁업자에게  귀속되는 것으로 

- 시멘틱 검색 + 키워드 검색

In [37]:
ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever,retriever],weights=[0.7,0.3])

docs = ensemble_retriever.invoke("가죽 운동화 세탁 보상방법")

for doc in docs:
    print(doc.metadata['case_id'],doc.page_content[:500])
    print('-----------')

01 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.

주 문 1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽  운동화, 색상 : 흰색) 1켤레를 반환한다.  2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원 을 지급한다.
이 유 1. 기초사실 가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색,  이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10.  피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피  마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였 으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신 청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세
-----------
32 ### 이 사건은 '식당에서 분실된 신발에 대한 배상 요구 에 대한 사례입니다.

의 비율에 의한 지연손해금을 가산하여 지급함이 상당하다.
[소비자분쟁해결기준에 따른 배상액]   운동화 사용 일수(2016. 9. 13. ~ 2016. 12. 13.) : 92일    배상비율표에 의한 배상비율 : 60%    배상액 : 운동화 구입 금액(109,000원) × 배상비율(60%) = 65,400원 [관련 법규 및 고시] 상법 제54조, 제152조, 소비자기본법 시행령 제9조, 소비자분쟁해 결기준 별표 II 품목별 해결기준 23. 세탁업, 별표 III 품목별 품질보증기간 및 부품보 유기간 이상과 같은 이유로 주문과 같이 결정한다.
-----------
01 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구 에 대한 사례입니다.

4 ● 2018 서비스·집단 분쟁조정 사례집 2 .  판   단 신청인은 피신청인의 세탁 후 신

#### SelfQuery

In [39]:
metadata_field_info =[
    AttributeInfo(
        name="case_id",description="사건번호",type="string"
    ),
    AttributeInfo(
        name="title",description="사건제목",type="string"
    ),
    AttributeInfo(
        name="description_date",description="결정일자",type="string"
    ),
]
self_retriever = SelfQueryRetriever.from_llm(
    llm=watson_llm,
    vectorstore=vectorstore,
    document_contents="소비자 분쟁 사례",
    metadata_field_info=metadata_field_info,
    structured_query_translator=ChromaTranslator()
)

docs = self_retriever.invoke("32번 사례 보여줘")

for doc in docs:
    print(doc.metadata['case_id'],doc.page_content[:500])
    print('-----------')


32 ### 이 사건은 '식당에서 분실된 신발에 대한 배상 요구 에 대한 사례입니다.

제1장 일 반 분 쟁 조 정  사 례 ( 서 비 스 ) 제1장 일반분쟁조정 사례(서비스) ● 89 피신청인이 운영하는 식당은 이용하는 고객들이 자신의 신발을 벗어 신발장에 둔 다음  식당 내로 들어가는 구조로 이루어져 있어 위 신발장은 고객들이 관리할 수 있는 영역 이라기보다 피신청인의 관리 영역이라고 볼 수 있고, 따라서 신발장에 둔 신발에 대해 서는 공중접객업자인 피신청인이 고객인 신청인으로부터 임치받았다고 봄이 상당하다.
피신청인은 고객으로부터 임치받은 물건이 멸실 또는 훼손되지 않도록 관리할 책임이  있음에도 시건장치를 갖춘 신발장을 설치하는 등의 조치 없이 식당 입구에 비닐봉지를  비치하고 주의 문구를 표시한 정도만으로는 임치 받은 신발의 보관에 관하여 주의를  게을리 하지 아니하였다고 보기에 부족하고, 「상법」제152조 제3항에 따라 고객의 휴 대물에 대해 책임이 없음을 알린 경우에도 물건의
-----------
32 ### 이 사건은 '식당에서 분실된 신발에 대한 배상 요구 에 대한 사례입니다.

살피건대, 공중접객업자는 자기 또는 그 사용인이 고객으로부터 임치(任置)받은 물건의  보관에 관하여 주의를 게을리하지 아니하였음을 증명하지 아니하면 그 물건의 멸실 또 는 훼손으로 인한 손해를 배상할 책임이 있다.
-----------
32 ### 이 사건은 '식당에서 분실된 신발에 대한 배상 요구 에 대한 사례입니다.

의 비율에 의한 지연손해금을 가산하여 지급함이 상당하다.
[소비자분쟁해결기준에 따른 배상액]   운동화 사용 일수(2016. 9. 13. ~ 2016. 12. 13.) : 92일    배상비율표에 의한 배상비율 : 60%    배상액 : 운동화 구입 금액(109,000원) × 배상비율(60%) = 65,400원 [관련 법규 및 고시] 상법 제54조, 제152조, 소비자기본법 시행령 제9조, 소비자분쟁해 결기준 별표 II 품목별 해결기준 23. 세탁업,


일반 검색 : k = 5
5 * 500 chunk =2500 자 전달
2500 전달 대신 질문과 관련있는 한 두 문장만 찾기 => LLMChainExtractor

In [40]:
compressor = LLMChainExtractor.from_llm(watson_llm)

compression_retriever = (ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever
))

docs = compression_retriever.invoke("식당에서 분실된 신발에 대한 손해배상은 어떻게 결정 되었나요?")

for doc in docs:
    print(doc.metadata['case_id'],doc.page_content[:500])
    print('-----------')

32 ### 이 사건은 '식당에서 분실된 신발에 대한 배상 요구 에 대한 사례입니다.

대물에 대해 책임이 없음을 알린 경우에도 물건의 멸실로 인한 손해를 배상할 책임을  면하지 못하므로, 신청인에게 운동화 분실에 따른 손해를 배상하여야 한다.
다만, 신청인은 자신의 신발을 누구나 접근할 수 있는 개방된 신발장에 비치하면서 분 실 가능성이 있음을 충분히 예상할 수 있었고, 피신청인이 비치한 비닐봉지를 이용하 여 자신의 신발을 다른 신발과 구분하는 등 주의를 기울일 필요가 있었음에도 어떠한  조치도 취하지 아니하였으므로, 이러한 신청인의 부주의를 고려하여 피신청인의 책임 을 50%로 제한하기로 한다.  그렇다면, 피신청인은 2017. 12. 26.까지 신청인에게 「소비자분쟁해결기준」에 따라  산정한 배상액 65,400원의 50%에 해당하는 32,700원을 지급하고, 만일 지급을 지체 하면 위 돈에 대하여 2017. 12. 27.부터 다 갚는 날까지 「상법」제54조에 따라 연 6% 의 비율에
-----------
32 ### 이 사건은 '식당에서 분실된 신발에 대한 배상 요구 에 대한 사례입니다.

살피건대, 공중접객업자는 자기 또는 그 사용인이 고객으로부터 임치(任置)받은 물건의  보관에 관하여 주의를 게을리하지 아니하였음을 증명하지 아니하면 그 물건의 멸실 또 는 훼손으로 인한 손해를 배상할 책임이 있다.
-----------
